v

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
data = pd.read_csv('/content/train (2).csv')

In [ ]:
null_columns_count = data.isnull().sum()[data.isnull().sum() > 0]
total_rows_of_null_columns = data.isnull().count()[data.isnull().sum() > 0]
null_percentage = (null_columns_count/total_rows_of_null_columns)*float(100)
missing_data = pd.concat([null_columns_count, null_percentage], axis=1, keys=['Total', 'Percentage'])
null_percentage.sort_values(ascending = False)

#Quitamos las columnas que tengan mas del 50% dde valores nulos
data_clean = data.drop((missing_data[missing_data['Percentage'] >50]).index,axis = 1)
data_clean1 = data_clean
null_columns_count = data_clean.isnull().sum()[data_clean.isnull().sum() > 0]
total_rows_of_null_columns = data_clean.isnull().count()[data_clean.isnull().sum() > 0]
null_percentage = (null_columns_count/total_rows_of_null_columns)*float(100)

null_percentage.sort_values(ascending = False)

#Cambiamos los valores nulos por 0
data_clean.fillna(0, inplace=True)
#Remplazamos los valores de las columnas objetos por enteros para que entren al modelo
for col in data_clean.columns:
    if data_clean[col].dtypes == object:
        for num, v in enumerate(data_clean[col].unique()):
            data_clean = data_clean.replace({col: {v : num}})


#Despúes de asignale un numero entero podemos quitar las columnas que no son muy significativas con respecto a (SalePrice)
#Utilizamos la correlación como medida.
for col in data_clean.columns:
    correlacion = data_clean['SalePrice'].corr(data_clean[col])
    #Si la correlación entre la columna y "SalePrice" es menor a 0.05, eliminamos la columna.
    if correlacion < 0.05:
        del data_clean[col]
data_clean.head()

data = pd.get_dummies(data, drop_first=True)
data = data.dropna()

<ipython-input-21-f9a9d86f73cb>:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_clean = data_clean.replace({col: {v : num}})


In [ ]:
# Correlación entre columnas numéricas
# ==============================================================================

def tidy_corr_matrix(corr_mat):
    '''
    Función para convertir una matriz de correlación de pandas en formato tidy
    '''
    corr_mat = corr_mat.stack().reset_index()
    corr_mat.columns = ['variable_1','variable_2','r']
    corr_mat = corr_mat.loc[corr_mat['variable_1'] != corr_mat['variable_2'], :]
    corr_mat['abs_r'] = np.abs(corr_mat['r'])
    corr_mat = corr_mat.sort_values('abs_r', ascending=False)

    return(corr_mat)



corr_matrix = data.select_dtypes(include=['float64', 'int']).corr(method='pearson')
tidy_corr_matrix(corr_matrix).loc[tidy_corr_matrix(corr_matrix)['variable_1'] == 'SalePrice'].head(12)

,variable_1,variable_2,r,abs_r
1410,SalePrice,OverallQual,0.797881,0.797881
1422,SalePrice,GrLivArea,0.705154,0.705154
1432,SalePrice,GarageCars,0.647034,0.647034
1433,SalePrice,GarageArea,0.619330,0.619330
1418,SalePrice,TotalBsmtSF,0.615612,0.615612
1419,SalePrice,1stFlrSF,0.607969,0.607969
1425,SalePrice,FullBath,0.566627,0.566627
1429,SalePrice,TotRmsAbvGrd,0.547067,0.547067
1412,SalePrice,YearBuilt,0.525394,0.525394
1413,SalePrice,YearRemodAdd,0.521253,0.521253


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

X = data[['OverallQual',	'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'FullBath',	'YearBuilt', 'YearRemodAdd']]
y = data['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y.values.reshape(-1,1),
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
modelo = LinearRegression()
modelo.fit(X = X_train, y = y_train)

LinearRegression()

In [ ]:
print("Coeficiente de determinación R^2:", modelo.score(X, y))

Coeficiente de determinación R^2: 0.7599633065783115


In [ ]:

X_train = X_train.astype(float)  # Convierte todo a tipo float
y_train = y_train.astype(float)  # Convierte el target a float

X_train = sm.add_constant(X_train, prepend=True)
modelo = sm.OLS(endog=y_train, exog=X_train,)
modelo = modelo.fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     381.7
Date:                Wed, 27 Nov 2024   Prob (F-statistic):          1.16e-262
Time:                        12:32:25   Log-Likelihood:                -10773.
No. Observations:                 896   AIC:                         2.156e+04
Df Residuals:                     888   BIC:                         2.160e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const        -1.202e+06   1.73e+05     -6.962   

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
sfs = SequentialFeatureSelector(
    modelo,
    n_features_to_select = 'auto',
    direction            = 'forward',
    scoring              = 'r2',
    cv                   = 5
)
sfs.fit(X_train, y_train)
sfs.get_feature_names_out().tolist()

KeyboardInterrupt: 